# 🧠 MS Lesion Segmentation: Clinical AI Dashboard
### Stage 2: Live Inference & Longitudinal Progression Demo

This notebook provides a professional, interactive interface for demonstrating automated MS lesion detection and tracking.

## 🛠 2.1 Minimal Environment Setup
Installing UI and medical imaging libraries.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel gradio matplotlib

import os, torch, SimpleITK as sitk, numpy as np
import matplotlib.pyplot as plt
from monai.networks.nets import UNet
from monai.inferers import sliding_window_inference
from scipy.ndimage import label
import gradio as gr

# Environment detection — identical pattern to training notebook
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ENVIRONMENT = "colab"
    MODEL_PATH  = "/content/drive/MyDrive/MS_Project/models/best_model.pth"
except ImportError:
    ENVIRONMENT = "local"
    MODEL_PATH  = "./models/best_model.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"Environment : {ENVIRONMENT}")
print(f"Device      : {device} ({DEVICE_NAME})")
print(f"Model path  : {MODEL_PATH}")
print("Dashboard Environment Ready.")

## 🧠 2.2 The Clinical Engine
This cell contains the 'One-Click' logic: Preprocessing, Inference, and Visual Overlays.

In [ ]:
import uuid, os as _os
from monai.networks.nets import BasicUNet

PATCH_SIZE  = (96, 96, 96)   # match training
OVERLAY_DIR = "./demo_outputs"
_os.makedirs(OVERLAY_DIR, exist_ok=True)


def get_model():
    return BasicUNet(
        spatial_dims=3, in_channels=1, out_channels=1,
        features=(32, 64, 128, 256, 512, 32),
        act="LEAKYRELU", norm="INSTANCE", dropout=0.1,
    )


def apply_skull_strip(sitk_img):
    otsu   = sitk.OtsuThreshold(sitk_img, 0, 1)
    filled = sitk.BinaryFillhole(otsu)
    closed = sitk.BinaryMorphologicalClosing(filled, [3, 3, 3])
    labeled = sitk.ConnectedComponent(closed)
    sorted_comp = sitk.RelabelComponent(labeled, sortByObjectSize=True)
    brain_mask = sitk.Equal(sorted_comp, 1)
    mask_float = sitk.Cast(brain_mask, sitk.sitkFloat32)
    return sitk.Multiply(sitk.Cast(sitk_img, sitk.sitkFloat32), mask_float)


def preprocess_for_demo(img_path):
    raw  = sitk.ReadImage(img_path)
    # N4 bias correction
    mask = sitk.OtsuThreshold(raw, 0, 1)
    cor  = sitk.N4BiasFieldCorrectionImageFilter()
    cor.SetMaximumNumberOfIterations([20, 20, 20])
    corrected = cor.Execute(raw, mask)
    # Skull strip
    stripped = apply_skull_strip(corrected)
    # RAS orientation
    img = sitk.DICOMOrient(stripped, "RAS")
    # 1mm isotropic resample
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing((1.0, 1.0, 1.0))
    orig_size, orig_spc = img.GetSize(), img.GetSpacing()
    new_size = [int(round(osz * ospc)) for osz, ospc in zip(orig_size, orig_spc)]
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    img = resampler.Execute(img)
    # Brain-masked Z-score normalization
    arr = sitk.GetArrayFromImage(img)
    brain_vox = arr[arr > 0]
    if brain_vox.size > 0:
        arr = (arr - brain_vox.mean()) / (brain_vox.std() + 1e-8)
    return arr  # return numpy array


def create_mosaic(img_vol, mask_vol):
    """3-panel axial/coronal/sagittal overlay at each volume midpoint."""
    D, H, W = img_vol.shape
    views = [
        (img_vol[D//2, :, :],  mask_vol[D//2, :, :],  "Axial"),
        (img_vol[:, H//2, :],  mask_vol[:, H//2, :],  "Coronal"),
        (img_vol[:, :, W//2],  mask_vol[:, :, W//2],  "Sagittal"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#0d1117')
    for ax, (img_sl, mask_sl, title) in zip(axes, views):
        ax.imshow(img_sl, cmap='gray', origin='lower')
        masked = np.ma.masked_where(mask_sl == 0, mask_sl)
        ax.imshow(masked, cmap='Reds', alpha=0.6, origin='lower')
        ax.set_title(title, color='white', fontsize=13, fontweight='bold')
        ax.axis('off')
    fig.tight_layout(pad=0.5)
    out = _os.path.join(OVERLAY_DIR, f"mosaic_{uuid.uuid4().hex[:8]}.png")
    plt.savefig(out, dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    return out


def create_change_mosaic(img_vol, new_mask, resolved_mask):
    """Green = new lesions, Blue = resolved lesions overlaid on T0."""
    D, H, W = img_vol.shape
    views = [
        (img_vol[D//2, :, :], new_mask[D//2, :, :], resolved_mask[D//2, :, :],  "Axial"),
        (img_vol[:, H//2, :], new_mask[:, H//2, :], resolved_mask[:, H//2, :],  "Coronal"),
        (img_vol[:, :, W//2], new_mask[:, :, W//2], resolved_mask[:, :, W//2],  "Sagittal"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#0d1117')
    for ax, (img_sl, new_sl, res_sl, title) in zip(axes, views):
        ax.imshow(img_sl, cmap='gray', origin='lower')
        ax.imshow(np.ma.masked_where(new_sl == 0, new_sl),
                  cmap='Greens', alpha=0.7, origin='lower')
        ax.imshow(np.ma.masked_where(res_sl == 0, res_sl),
                  cmap='Blues', alpha=0.7, origin='lower')
        ax.set_title(title, color='white', fontsize=13, fontweight='bold')
        ax.axis('off')
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(color='lime', label='New lesions'),
                        Patch(color='cornflowerblue', label='Resolved lesions')],
               loc='lower center', ncol=2, framealpha=0, labelcolor='white', fontsize=11)
    fig.tight_layout(pad=0.5)
    out = _os.path.join(OVERLAY_DIR, f"change_{uuid.uuid4().hex[:8]}.png")
    plt.savefig(out, dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    return out


def run_lesion_stats(mask):
    """Volume, count, and small/medium/large breakdown."""
    from scipy.ndimage import label as cc_label
    labeled, n = cc_label(mask)
    vol = int(mask.sum())
    size_tp = {"small": 0, "medium": 0, "large": 0}
    for comp_id in range(1, n + 1):
        sz = int((labeled == comp_id).sum())
        if sz < 10:       size_tp["small"] += 1
        elif sz <= 100:   size_tp["medium"] += 1
        else:             size_tp["large"] += 1
    return vol, n, size_tp


def remove_fp_noise(mask, min_size=5):
    """Remove connected components smaller than min_size voxels."""
    from scipy.ndimage import label as cc_label
    labeled, n = cc_label(mask)
    cleaned = np.zeros_like(mask, dtype=bool)
    for comp_id in range(1, n + 1):
        if (labeled == comp_id).sum() >= min_size:
            cleaned[labeled == comp_id] = True
    return cleaned


def segment_volume(arr):
    t = torch.tensor(arr).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        if device.type == "cuda":
            with torch.amp.autocast('cuda'):
                prob = sliding_window_inference(t, PATCH_SIZE, 4, model)
        else:
            prob = sliding_window_inference(t, PATCH_SIZE, 4, model)
    prob_np = prob.sigmoid().cpu().numpy().squeeze()
    mask = (prob_np > 0.5).astype(bool)
    mask = remove_fp_noise(mask, min_size=5)
    return mask, prob_np


def compute_confidence_metrics(mask, prob_np):
    """Tier 1: always-available confidence stats from model output."""
    if mask.sum() == 0:
        return {"mean_confidence": 0.0, "brain_coverage_pct": 0.0, "high_conf_pct": 0.0}
    lesion_probs = prob_np[mask]
    brain_voxels = (prob_np > 0.01).sum()   # rough brain mask
    return {
        "mean_confidence":    float(lesion_probs.mean()),
        "brain_coverage_pct": float(mask.sum() / max(brain_voxels, 1) * 100),
        "high_conf_pct":      float((lesion_probs > 0.8).mean() * 100),
    }


def compute_accuracy_metrics(pred_mask, gt_mask):
    """Tier 2: requires ground truth mask."""
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    tp = (pred & gt).sum()
    fp = (pred & ~gt).sum()
    fn = (~pred & gt).sum()
    tn = (~pred & ~gt).sum()
    dice        = 2*tp / (2*tp + fp + fn + 1e-8)
    sensitivity = tp / (tp + fn + 1e-8)
    precision   = tp / (tp + fp + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    return {"dice": float(dice), "sensitivity": float(sensitivity),
            "precision": float(precision), "specificity": float(specificity)}


def run_ai_analysis(file_obj, gt_file_obj=None):
    if file_obj is None:
        return None, "Please upload a NIfTI file (.nii or .nii.gz)."
    arr = preprocess_for_demo(file_obj.name)
    mask, prob_np = segment_volume(arr)
    vol, count, sizes = run_lesion_stats(mask)
    conf = compute_confidence_metrics(mask, prob_np)

    acc_section = ""
    if gt_file_obj is not None:
        import nibabel as nib
        from skimage.transform import resize as sk_resize
        gt_nib = nib.load(gt_file_obj.name)
        gt_arr = (gt_nib.get_fdata() > 0.5).astype(bool)
        if gt_arr.shape != mask.shape:
            gt_arr = sk_resize(gt_arr, mask.shape, order=0, anti_aliasing=False).astype(bool)
        acc = compute_accuracy_metrics(mask, gt_arr)
        acc_section = f"""
### Segmentation Accuracy (vs Ground Truth)
| Metric | Score |
|--------|-------|
| **Dice Score** | **{acc['dice']:.3f}** |
| Sensitivity (Recall) | {acc['sensitivity']:.3f} |
| Precision | {acc['precision']:.3f} |
| Specificity | {acc['specificity']:.3f} |
"""

    report = f"""
## AI Analysis Report
| Metric | Value |
|--------|-------|
| Total Lesion Volume | **{vol:.0f} mm³** |
| Total Lesion Count  | **{count}** |
| Small (<10 vox)     | {sizes['small']} |
| Medium (10-100 vox) | {sizes['medium']} |
| Large (>100 vox)    | {sizes['large']} |

### Model Confidence
| Metric | Value |
|--------|-------|
| Mean Lesion Confidence | {conf['mean_confidence']:.1%} |
| High-Confidence Voxels (>80%) | {conf['high_conf_pct']:.1f}% of lesion |
| Brain Coverage | {conf['brain_coverage_pct']:.3f}% |
{acc_section}
*Processed on {DEVICE_NAME}*
"""
    mosaic_path = create_mosaic(arr, mask)
    return mosaic_path, report


def run_longitudinal_analysis(t0_file, t1_file):
    if t0_file is None or t1_file is None:
        return None, None, "Please upload both Baseline (T0) and Follow-up (T1) scans."

    arr_t0 = preprocess_for_demo(t0_file.name)
    arr_t1 = preprocess_for_demo(t1_file.name)

    mask_t0, _ = segment_volume(arr_t0)
    mask_t1, _ = segment_volume(arr_t1)

    # Rigid registration: align T1 into T0 space
    fixed  = sitk.GetImageFromArray(arr_t0.astype(np.float32))
    moving = sitk.GetImageFromArray(arr_t1.astype(np.float32))
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetOptimizerAsGradientDescent(learningRate=1.0, numberOfIterations=100)
    reg.SetInitialTransform(sitk.CenteredTransformInitializer(
        fixed, moving, sitk.Euler3DTransform()))
    reg.SetInterpolator(sitk.sitkLinear)
    tx = reg.Execute(fixed, moving)

    # Resample T1 mask into T0 space
    mask_t1_sitk = sitk.GetImageFromArray(mask_t1.astype(np.float32))
    mask_t1_sitk.CopyInformation(moving)
    mask_t1_reg_sitk = sitk.Resample(mask_t1_sitk, fixed, tx,
                                      sitk.sitkNearestNeighbor, 0.0)
    mask_t1_reg = sitk.GetArrayFromImage(mask_t1_reg_sitk).astype(bool)

    new_lesions      = mask_t1_reg & ~mask_t0
    resolved_lesions = mask_t0 & ~mask_t1_reg

    img_t0_mosaic = create_mosaic(arr_t0, mask_t0)
    change_vis    = create_change_mosaic(arr_t0, new_lesions, resolved_lesions)

    vol_t0, cnt_t0, _ = run_lesion_stats(mask_t0)
    vol_t1, cnt_t1, _ = run_lesion_stats(mask_t1_reg)
    _, new_cnt, _     = run_lesion_stats(new_lesions)
    _, res_cnt, _     = run_lesion_stats(resolved_lesions)

    report  = "## Longitudinal Progression Report\n\n"
    report += "| Metric | Baseline (T0) | Follow-up (T1) | Change |\n|---|---|---|---|\n"
    report += f"| Lesion Volume | {vol_t0} mm³ | {vol_t1} mm³ | {vol_t1 - vol_t0:+d} mm³ |\n"
    report += f"| Lesion Count  | {cnt_t0} | {cnt_t1} | {cnt_t1 - cnt_t0:+d} |\n"
    report += f"\n**New lesions detected:** {new_cnt} &nbsp;&nbsp; "
    report += f"**Resolved lesions:** {res_cnt}\n"
    report += f"\n*Registration + segmentation on {DEVICE_NAME}*"

    return img_t0_mosaic, change_vis, report


print("Clinical Engine Logic defined.")

## 🚀 2.3 Interactive Dashboard (Gradio)
Run this cell to launch your presentation UI.

In [ ]:
model = get_model().to(device)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"WARNING: Model not found at {MODEL_PATH}. Run training first.")

css = """
body { font-family: 'Segoe UI', sans-serif; }
.gr-button-primary { background: #2563eb !important; }
"""

with gr.Blocks(title="MS Lesion AI", theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown("# MS Lesion Segmentation — AI Diagnostic Dashboard")
    gr.Markdown("3D U-Net trained on MSLesSeg + Mendeley + Long-MR-MS datasets.")

    with gr.Tab("Single-Scan Analysis"):
        gr.Markdown("Upload a **FLAIR MRI** (`.nii` or `.nii.gz`) to detect MS lesions. Optionally upload a ground-truth mask to see accuracy metrics.")
        with gr.Row():
            with gr.Column(scale=1):
                s_input  = gr.File(label="FLAIR MRI (NIfTI)", file_types=[".nii", ".gz"])
                gt_input = gr.File(label="Ground Truth Mask — optional (.nii / .nii.gz)", file_types=[".nii", ".gz"])
                s_btn    = gr.Button("Analyze Lesions", variant="primary")
            with gr.Column(scale=2):
                s_img    = gr.Image(label="Axial | Coronal | Sagittal Overlay")
                s_report = gr.Markdown()
        s_btn.click(run_ai_analysis, inputs=[s_input, gt_input], outputs=[s_img, s_report])

    with gr.Tab("Longitudinal Tracking"):
        gr.Markdown("Upload **Baseline (T0)** and **Follow-up (T1)** FLAIR scans to detect new disease activity.")
        gr.Markdown("> Scans are rigidly registered before comparison — no manual alignment needed.")
        with gr.Row():
            t0_in = gr.File(label="Baseline Scan — T0 (.nii / .nii.gz)", file_types=[".nii", ".gz"])
            t1_in = gr.File(label="Follow-up Scan — T1 (.nii / .nii.gz)", file_types=[".nii", ".gz"])
        l_btn = gr.Button("Detect Progression", variant="primary")
        with gr.Row():
            l_img_t0  = gr.Image(label="Baseline Segmentation (T0)")
            l_img_chg = gr.Image(label="Change Map  [Green = New   Blue = Resolved]")
        l_report = gr.Markdown()
        l_btn.click(run_longitudinal_analysis,
                    inputs=[t0_in, t1_in],
                    outputs=[l_img_t0, l_img_chg, l_report])

demo.launch(share=True)